#3

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

In [9]:
dataset = np.load('dataset6.npz', allow_pickle=True)
X_train = dataset['X_train']
X_test = dataset['X_test']
y_train = dataset['y_train']
y_test = dataset['y_test']

In [11]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(400, 4)
(100, 4)
(400,)
(100,)


In [34]:
print(X_train[:5])
print(y_train[:5])
print(X_test[:5])
print(y_test[:5])

[[-4.7992244140674 'E' -562.4562871001353 -873.0664483997828]
 [6.659631281359991 'E' -452.10797209694465 414.45177770448134]
 [-0.5943567889576173 'B' 842.8584952946348 -1775.8854238863546]
 [-4.186307455511265 'B' -88.58441053800242 -249.11611580361767]
 [-2.6306888170211016 'B' -1089.9702133952935 -449.48522061616904]]
[1 1 0 0 0]
[[10.21603202946599 'E' -1090.3207244165387 -81.12488990905761]
 [-7.744748029973359 'A' 1463.3869155501964 581.4913274526882]
 [-3.7086984308964435 'B' -1253.3416576349189 2505.3325855035014]
 [-0.5877243471779412 'C' 1392.6655541910623 -687.0930029042455]
 [0.4984740418978886 'C' 507.6855721452738 259.87538918315715]]
[1 2 0 2 1]


In [13]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

ValueError: could not convert string to float: 'E'

In [14]:
print("Data type:", X_train.dtype)

print("Categories in training:", np.unique(X_train[:, 1]))
print("Categories in testing:", np.unique(X_test[:, 1]))

X_numeric = X_train[:, [0, 2, 3]].astype(float)

print("\nNumerical feature statistics:")
print("Minimum:", X_numeric.min(axis=0))
print("Maximum:", X_numeric.max(axis=0))
print("Mean:", X_numeric.mean(axis=0))
print("Standard deviation:", X_numeric.std(axis=0))

print("\nTraining class distribution:")
print(np.unique(y_train, return_counts=True))

print("\nTesting class distribution:")
print(np.unique(y_test, return_counts=True))


Data type: object
Categories in training: ['A' 'B' 'C' 'D' 'E']
Categories in testing: ['A' 'B' 'C' 'D' 'E']

Numerical feature statistics:
Minimum: [  -14.16638012 -2760.18297219 -3139.12957486]
Maximum: [  15.41596167 3253.66675534 2529.13568398]
Mean: [ -1.55794332  -3.73769246 -17.89643833]
Standard deviation: [  5.64069082 980.18396081 938.58041024]

Training class distribution:
(array([0, 1, 2]), array([133, 134, 133]))

Testing class distribution:
(array([0, 1, 2]), array([34, 33, 33]))


In [37]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), [1])
    ],
    remainder='passthrough'
)
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_encoded, y_train)
y_pred = model.predict(X_test_encoded)
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy:", accuracy)

Test accuracy: 0.41


In [35]:
majority = np.bincount(y_train).argmax()
baseline_accuracy = np.mean(y_test == majority)
print(baseline_accuracy)

0.33


B

In [41]:
a = X_train[0, [0, 2, 3]].astype(float)
b = X_train[1, [0, 2, 3]].astype(float)
sq = (a - b) ** 2
print(sq / sq.sum())

[7.86254432e-05 7.29141837e-03 9.92629956e-01]


In [42]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), [0, 2, 3]),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), [1])
    ]
)
model = Pipeline([
    ('preprocessing', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy after scaling:", accuracy)

Test accuracy after scaling: 0.95


In [46]:
from sklearn.model_selection import cross_val_score

for k in [1, 3, 5, 7, 9, 11, 15, 21, 25, 31, 41]:
    m = Pipeline([('preprocessing', preprocessor),
                  ('knn', KNeighborsClassifier(n_neighbors=k))])
    print(k, cross_val_score(m, X_train, y_train, cv=5).mean())

1 0.8525
3 0.885
5 0.8875
7 0.8899999999999999
9 0.9024999999999999
11 0.9
15 0.9075
21 0.915
25 0.9199999999999999
31 0.9199999999999999
41 0.9025000000000001


In [47]:
best_k = 25
final = Pipeline([('preprocessing', preprocessor),
                  ('knn', KNeighborsClassifier(n_neighbors=best_k))])
final.fit(X_train, y_train)
print("Test accuracy:", accuracy_score(y_test, final.predict(X_test)))

Test accuracy: 0.96
